In [ ]:
!cp /content/drive/MyDrive/data.zip /content/data.zip
!unzip -q /content/data.zip -d /content/data/


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
for cls in ["Normal", "Pneumonia", "ILD"]:
    path = f"/content/data/data/{cls}"
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        print(f"  {cls}: {count} images")
    else:
        print(f"  ⚠ {cls} not found!")

  Normal: 60361 images
  Pneumonia: 322 images
  ILD: 11166 images


In [ ]:
import os, json, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import Counter

# Config
CLASSES = ["Normal", "Pneumonia", "ILD"]
NUM_CLASSES = 3
IMG_SIZE = 224
BATCH = 64
EPOCHS = 25
LR = 1e-4
PATIENCE = 5  # early stopping patience
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data/data/"
OUT_DIR = "./model"
os.makedirs(OUT_DIR, exist_ok=True)


# Dataset
class CXRDataset(Dataset):
    def __init__(self, paths, labels, tfm=None):
        self.paths, self.labels, self.tfm = paths, labels, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.labels[i]


train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def load_paths(data_dir):
    paths, labels = [], []
    for i, cls in enumerate(CLASSES):
        d = os.path.join(data_dir, cls)
        if not os.path.isdir(d):
            print(f"⚠ {d} not found — skipping"); continue
        files = [f for f in os.listdir(d) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        paths += [os.path.join(d, f) for f in files]
        labels += [i] * len(files)
        print(f"  {cls}: {len(files)} images")
    return paths, labels


# Model
class LungClassifier(nn.Module):
    def __init__(self, n_cls=NUM_CLASSES):
        super().__init__()
        self.backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        n_feat = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(n_feat, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_cls)
        )
    def forward(self, x):
        return self.head(self.backbone(x))


# Grad-CAM (for local testing)
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.grads = self.acts = None
        layer = model.backbone.features.denseblock4.denselayer16.conv2
        layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0].detach()))

    def generate(self, x, cls_idx=None):
        self.model.eval()
        out = self.model(x)
        if cls_idx is None: cls_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, cls_idx].backward()
        w = self.grads.mean(dim=[2, 3], keepdim=True)
        cam = torch.relu((w * self.acts).sum(1, keepdim=True))
        cam = nn.functional.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        probs = torch.softmax(out, 1)[0].detach().cpu().numpy()
        return cam.cpu().numpy(), cls_idx, probs


# Training
def train():
    print(f"Device: {DEVICE}\nLoading data from {DATA_DIR}...")
    paths, labels = load_paths(DATA_DIR)
    if not paths:
        print("No data found! Run organize_dataset.py first."); return

    tr_p, va_p, tr_l, va_l = train_test_split(
        paths, labels, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Train: {len(tr_p)} | Val: {len(va_p)}")

    # Weighted sampler for class imbalance
    counts = Counter(tr_l)
    print(f"Train class counts: { {CLASSES[k]: v for k, v in counts.items()} }")
    weights = [1.0 / counts[l] for l in tr_l]
    sampler = WeightedRandomSampler(weights, len(weights))

    use_cuda = torch.cuda.is_available()
    tr_dl = DataLoader(CXRDataset(tr_p, tr_l, train_tfm), BATCH, sampler=sampler, num_workers=0, pin_memory=use_cuda)
    va_dl = DataLoader(CXRDataset(va_p, va_l, val_tfm), BATCH, shuffle=False, num_workers=0, pin_memory=use_cuda)

    model = LungClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_acc = 0
    patience_counter = 0

    for epoch in range(EPOCHS):
        # Train
        model.train()
        running_loss, correct, total = 0, 0, 0
        for batch_idx, (imgs, lbls) in enumerate(tr_dl):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
            total += imgs.size(0)

            # Progress indicator every 20 batches
            if (batch_idx + 1) % 20 == 0:
                print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(tr_dl)} | "
                      f"Loss: {loss.item():.4f}", end='\r')

        # Validate
        model.eval()
        va_correct, va_total, va_preds, va_true = 0, 0, [], []
        with torch.no_grad():
            for imgs, lbls in va_dl:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                preds = model(imgs).argmax(1)
                va_correct += (preds == lbls).sum().item()
                va_total += imgs.size(0)
                va_preds.extend(preds.cpu().tolist())
                va_true.extend(lbls.cpu().tolist())

        tr_acc = correct / total
        va_acc = va_correct / va_total
        tr_loss = running_loss / total
        scheduler.step(1 - va_acc)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1}/{EPOCHS}  loss={tr_loss:.4f}  "
              f"train_acc={tr_acc:.4f}  val_acc={va_acc:.4f}  lr={current_lr:.6f}")

        # Save best + early stopping
        if va_acc > best_acc:
            best_acc = va_acc
            patience_counter = 0
            torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
            print(f"  ✓ Saved best model (val_acc={va_acc:.4f})")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{PATIENCE})")
            if patience_counter >= PATIENCE:
                print(f"\n⏹ Early stopping — no improvement for {PATIENCE} epochs")
                break

    # Final classification report
    print(f"\n{'='*50}")
    print(f"Best validation accuracy: {best_acc:.4f}")
    print(f"{'='*50}")
    print("\nClassification Report (last epoch):")
    print(classification_report(va_true, va_preds, target_names=CLASSES))

    # Export to ONNX
    print("Exporting to ONNX...")
    model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location="cpu"))
    model.eval().cpu()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(
        model, dummy, f"{OUT_DIR}/model.onnx",
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17
    )
    with open(f"{OUT_DIR}/classes.json", "w") as f:
        json.dump(CLASSES, f)

    fsize = os.path.getsize(f"{OUT_DIR}/model.onnx") / 1e6
    print(f"\n✓ Exported model.onnx ({fsize:.1f} MB)")
    print(f"✓ classes.json: {CLASSES}")
    print(f"\nNext steps:")
    print(f"  1. Upload to S3:  aws s3 cp model/model.onnx s3://your-bucket/model/model.onnx")
    print(f"  2. Upload to S3:  aws s3 cp model/classes.json s3://your-bucket/model/classes.json")
    print(f"  3. Deploy:        bash deploy.sh")


if __name__ == "__main__":
    train()

Device: cuda
Loading data from ./data/data/...
  Normal: 60361 images
  Pneumonia: 322 images
  ILD: 11166 images
Train: 57479 | Val: 14370
Train class counts: {'Normal': 48289, 'ILD': 8933, 'Pneumonia': 257}
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 270MB/s]


Epoch 1/25  loss=0.6281  train_acc=0.6634  val_acc=0.6115  lr=0.000100
  ✓ Saved best model (val_acc=0.6115)
Epoch 2/25  loss=0.4566  train_acc=0.7441  val_acc=0.6281  lr=0.000100
  ✓ Saved best model (val_acc=0.6281)
Epoch 3/25  loss=0.4387  train_acc=0.7538  val_acc=0.6335  lr=0.000100
  ✓ Saved best model (val_acc=0.6335)
Epoch 4/25  loss=0.4211  train_acc=0.7684  val_acc=0.7200  lr=0.000100
  ✓ Saved best model (val_acc=0.7200)
Epoch 5/25  loss=0.4076  train_acc=0.7822  val_acc=0.5619  lr=0.000100
  No improvement (1/5)
Epoch 6/25  loss=0.3952  train_acc=0.7913  val_acc=0.6951  lr=0.000100
  No improvement (2/5)
Epoch 7/25  loss=0.3741  train_acc=0.8093  val_acc=0.7131  lr=0.000100
  No improvement (3/5)
Epoch 8/25  loss=0.3578  train_acc=0.8218  val_acc=0.6176  lr=0.000050
  No improvement (4/5)
Epoch 9/25  loss=0.3063  train_acc=0.8560  val_acc=0.6862  lr=0.000050
  No improvement (5/5)

⏹ Early stopping — no improvement for 5 epochs

Best validation accuracy: 0.7200

Classificat

ModuleNotFoundError: No module named 'onnxscript'

In [ ]:
!pip install onnxscript onnx

In [ ]:
import torch, torch.nn as nn, json, os
from torchvision import models

OUT_DIR = "/content/model_v3"
IMG_SIZE = 224

# Recreate model architecture
class LungClassifier(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.densenet121(weights=None)
        n_feat = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(n_feat, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_cls)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

# Load trained weights
model = LungClassifier()
model.load_state_dict(torch.load(f"/content/model/best_model.pt", map_location="cpu"))
model.eval().cpu()
print("✓ Model loaded")

# Export to ONNX
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model, dummy, f"{OUT_DIR}/model.onnx",
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17
)
with open(f"{OUT_DIR}/classes.json", "w") as f:
    json.dump(["Normal", "Pneumonia", "ILD"], f)

fsize = os.path.getsize(f"{OUT_DIR}/model.onnx") / 1e6
print(f"✓ model.onnx ({fsize:.1f} MB)")
print(f"✓ classes.json saved")

In [ ]:
import os, json, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import time, random

# ─── Config ───
CLASSES = ["Normal", "Pneumonia", "ILD"]
IMG_SIZE = 224
BATCH = 64
EPOCHS = 25
LR = 1e-4
PATIENCE = 7
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/data/data"
OUT_DIR = "/content/model_v2"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── Dataset with stronger augmentation ───
class CXRDataset(Dataset):
    def __init__(self, paths, labels, tfm=None):
        self.paths, self.labels, self.tfm = paths, labels, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.labels[i]

# Stronger augmentation to help minority classes
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ─── Load and BALANCE data ───
def load_paths(data_dir, max_per_class=5000):
    paths, labels = [], []
    for i, cls in enumerate(CLASSES):
        d = os.path.join(data_dir, cls)
        if not os.path.isdir(d): continue
        files = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.png')]
        # Cap large classes to reduce imbalance
        if len(files) > max_per_class:
            random.seed(42)
            files = random.sample(files, max_per_class)
        paths += files
        labels += [i] * len(files)
        print(f"  {cls}: {len(files)} images")
    return paths, labels

print(f"Device: {DEVICE}")
print("Loading balanced data...")
paths, labels = load_paths(DATA_DIR, max_per_class=3000)

tr_p, va_p, tr_l, va_l = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Train: {len(tr_p)} | Val: {len(va_p)}")
print(f"Train counts: { {CLASSES[k]: v for k, v in Counter(tr_l).items()} }")
print(f"Val counts:   { {CLASSES[k]: v for k, v in Counter(va_l).items()} }")

# Weighted sampler — aggressively oversample Pneumonia
counts = Counter(tr_l)
weights = [1.0 / counts[l] for l in tr_l]
sampler = WeightedRandomSampler(weights, len(weights))

tr_dl = DataLoader(CXRDataset(tr_p, tr_l, train_tfm), BATCH, sampler=sampler, num_workers=2, pin_memory=True)
va_dl = DataLoader(CXRDataset(va_p, va_l, val_tfm), BATCH, shuffle=False, num_workers=2, pin_memory=True)

# ─── Model: EfficientNetB0 (better than DenseNet for this) ───
class LungClassifierV2(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        n_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n_feat, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_cls)
        )
    def forward(self, x):
        return self.backbone(x)

model = LungClassifierV2().to(DEVICE)

# ─── Class-weighted loss — Pneumonia errors cost MORE ───
class_counts = Counter(tr_l)
total = sum(class_counts.values())
class_weights = torch.tensor([total / (3 * class_counts[i]) for i in range(3)], dtype=torch.float32).to(DEVICE)
print(f"Loss weights: Normal={class_weights[0]:.2f}, Pneumonia={class_weights[1]:.2f}, ILD={class_weights[2]:.2f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ─── Train ───
best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

print(f"\nTraining on {DEVICE} for up to {EPOCHS} epochs...\n")
start = time.time()

for epoch in range(EPOCHS):
    ep_start = time.time()
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, lbls in tr_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)

    model.eval()
    va_correct, va_total, va_preds, va_true = 0, 0, [], []
    with torch.no_grad():
        for imgs, lbls in va_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            preds = model(imgs).argmax(1)
            va_correct += (preds == lbls).sum().item()
            va_total += imgs.size(0)
            va_preds.extend(preds.cpu().tolist())
            va_true.extend(lbls.cpu().tolist())

    tr_acc = correct / total
    va_acc = va_correct / va_total
    tr_loss = running_loss / total
    scheduler.step()

    # Calculate macro F1 — better metric for imbalanced data
    report = classification_report(va_true, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(macro_f1)

    ep_time = time.time() - ep_start
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_acc={va_acc:.4f}  macro_f1={macro_f1:.4f}  time={ep_time:.0f}s")

    # Save best by macro F1 (not accuracy — accuracy is misleading with imbalance)
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
        print(f"  ✓ Saved best model (macro_f1={macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

total_time = time.time() - start
print(f"\nTotal time: {total_time/60:.1f} min | Best macro F1: {best_f1:.4f}")

# ─── Final report ───
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location=DEVICE))
model.eval()
va_preds, va_true = [], []
with torch.no_grad():
    for imgs, lbls in va_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        va_preds.extend(model(imgs).argmax(1).cpu().tolist())
        va_true.extend(lbls.cpu().tolist())

print("\n" + "="*50)
print("Final Classification Report:")
print("="*50)
print(classification_report(va_true, va_preds, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(va_true, va_preds)
print("Confusion Matrix:")
print(f"{'':>12} {'Normal':>8} {'Pneumonia':>10} {'ILD':>8}")
for i, cls in enumerate(CLASSES):
    print(f"{cls:>12} {cm[i][0]:>8} {cm[i][1]:>10} {cm[i][2]:>8}")

In [ ]:
import torch, torch.nn as nn, json, os
from torchvision import models

OUT_DIR = "/content/model"
IMG_SIZE = 224

# Recreate model architecture
class LungClassifier(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.densenet121(weights=None)
        n_feat = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(n_feat, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_cls)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

# Load trained weights
model = LungClassifier()
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location="cpu"))
model.eval().cpu()
print("✓ Model loaded")

# Export to ONNX
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model, dummy, f"{OUT_DIR}/model_v2.onnx",
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17
)
with open(f"{OUT_DIR}/classes_v2.json", "w") as f:
    json.dump(["Normal", "Pneumonia", "ILD"], f)

fsize = os.path.getsize(f"{OUT_DIR}/model_v2.onnx") / 1e6
print(f"✓ model.onnx ({fsize:.1f} MB)")
print(f"✓ classes.json saved")

In [ ]:
import os, json, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import time, random, copy

CLASSES = ["Normal", "Pneumonia", "ILD"]
IMG_SIZE = 224
BATCH = 64
EPOCHS = 40
LR = 3e-4
PATIENCE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/data/data"
OUT_DIR = "/content/model_v3"
os.makedirs(OUT_DIR, exist_ok=True)

class CXRDataset(Dataset):
    def __init__(self, paths, labels, tfm=None):
        self.paths, self.labels, self.tfm = paths, labels, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.labels[i]

# Aggressive augmentation
train_tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ─── KEY CHANGE: Balance by oversampling Pneumonia at the file level ───
def load_balanced(data_dir):
    class_paths = {}
    for i, cls in enumerate(CLASSES):
        d = os.path.join(data_dir, cls)
        files = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.png')]
        class_paths[cls] = files
        print(f"  {cls}: {len(files)} raw images")

    # Target: 2000 per class
    # Normal & ILD: downsample to 2000
    # Pneumonia: repeat to reach 2000
    TARGET = 2000
    paths, labels = [], []
    for i, cls in enumerate(CLASSES):
        files = class_paths[cls]
        random.seed(42)
        if len(files) >= TARGET:
            files = random.sample(files, TARGET)
        else:
            # Oversample: repeat images until we reach TARGET
            repeats = TARGET // len(files)
            remainder = TARGET % len(files)
            files = files * repeats + random.sample(files, remainder)
        paths += files
        labels += [i] * len(files)
        print(f"  {cls}: {len(files)} after balancing")
    return paths, labels

print(f"Device: {DEVICE}")
print("Loading and balancing data...")
paths, labels = load_balanced(DATA_DIR)

tr_p, va_p, tr_l, va_l = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"\nTrain: {len(tr_p)} | Val: {len(va_p)}")

# Still use weighted sampler on top of balanced data
counts = Counter(tr_l)
weights = [1.0 / counts[l] for l in tr_l]
sampler = WeightedRandomSampler(weights, len(weights))

tr_dl = DataLoader(CXRDataset(tr_p, tr_l, train_tfm), BATCH, sampler=sampler, num_workers=2, pin_memory=True)
va_dl = DataLoader(CXRDataset(va_p, va_l, val_tfm), BATCH, shuffle=False, num_workers=2, pin_memory=True)

# ─── EfficientNetB2 (slightly bigger, better features) ───
class LungClassifierV3(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        n_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n_feat, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, n_cls)
        )
    def forward(self, x):
        return self.backbone(x)

model = LungClassifierV3().to(DEVICE)

# ─── Freeze backbone first, train head only for 5 epochs ───
for param in model.backbone.features.parameters():
    param.requires_grad = False

# Phase 1: Train head only
print("\n── Phase 1: Training head only (5 epochs) ──")
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

for epoch in range(5):
    model.train()
    correct, total = 0, 0
    for imgs, lbls in tr_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    print(f"  Head epoch {epoch+1}/5  train_acc={correct/total:.4f}")

# Phase 2: Unfreeze everything, fine-tune with lower LR
print("\n── Phase 2: Fine-tuning entire model ──")
for param in model.backbone.features.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

start = time.time()
for epoch in range(EPOCHS):
    ep_start = time.time()
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, lbls in tr_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)

    model.eval()
    va_preds, va_true = [], []
    with torch.no_grad():
        for imgs, lbls in va_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            va_preds.extend(model(imgs).argmax(1).cpu().tolist())
            va_true.extend(lbls.cpu().tolist())

    tr_acc = correct / total
    va_acc = sum(p == t for p, t in zip(va_preds, va_true)) / len(va_true)
    report = classification_report(va_true, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    scheduler.step()

    history["train_loss"].append(running_loss / total)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(macro_f1)

    ep_time = time.time() - ep_start
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={running_loss/total:.4f}  train_acc={tr_acc:.4f}  "
          f"val_acc={va_acc:.4f}  macro_f1={macro_f1:.4f}  time={ep_time:.0f}s")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
        print(f"  ✓ Saved best model (macro_f1={macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

print(f"\nTotal time: {(time.time()-start)/60:.1f} min | Best macro F1: {best_f1:.4f}")

# ─── Final report ───
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location=DEVICE))
model.eval()
va_preds, va_true = [], []
with torch.no_grad():
    for imgs, lbls in va_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        va_preds.extend(model(imgs).argmax(1).cpu().tolist())
        va_true.extend(lbls.cpu().tolist())

print("\n" + "="*50)
print("Final Classification Report:")
print("="*50)
print(classification_report(va_true, va_preds, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(va_true, va_preds)
print("Confusion Matrix:")
print(f"{'':>12} {'Normal':>8} {'Pneumonia':>10} {'ILD':>8}")
for i, cls in enumerate(CLASSES):
    print(f"{cls:>12} {cm[i][0]:>8} {cm[i][1]:>10} {cm[i][2]:>8}")

Device: cuda
Loading and balancing data...
  Normal: 60361 raw images
  Pneumonia: 322 raw images
  ILD: 11166 raw images
  Normal: 2000 after balancing
  Pneumonia: 2000 after balancing
  ILD: 2000 after balancing

Train: 4800 | Val: 1200
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 356MB/s]


── Phase 1: Training head only (5 epochs) ──


  Head epoch 1/5  train_acc=0.3767
  Head epoch 2/5  train_acc=0.4117
  Head epoch 3/5  train_acc=0.3927
  Head epoch 4/5  train_acc=0.4029
  Head epoch 5/5  train_acc=0.4117

── Phase 2: Fine-tuning entire model ──
Epoch  1/40  loss=1.0677  train_acc=0.4285  val_acc=0.4550  macro_f1=0.4427  time=37s
  ✓ Saved best model (macro_f1=0.4427)
Epoch  2/40  loss=1.0157  train_acc=0.4792  val_acc=0.4775  macro_f1=0.4777  time=37s
  ✓ Saved best model (macro_f1=0.4777)
Epoch  3/40  loss=0.9720  train_acc=0.5090  val_acc=0.5158  macro_f1=0.5032  time=37s
  ✓ Saved best model (macro_f1=0.5032)
Epoch  4/40  loss=0.9010  train_acc=0.5681  val_acc=0.5458  macro_f1=0.5457  time=37s
  ✓ Saved best model (macro_f1=0.5457)
Epoch  5/40  loss=0.8313  train_acc=0.5983  val_acc=0.5800  macro_f1=0.5733  time=37s
  ✓ Saved best model (macro_f1=0.5733)
Epoch  6/40  loss=0.7500  train_acc=0.6471  val_acc=0.6225  macro_f1=0.6209  time=37s
  ✓ Saved best model (macro_f1=0.6209)
Epoch  7/40  loss=0.7147  train_ac

In [ ]:
# Export V3 to ONNX
import torch, torch.nn as nn, json, os, shutil
from torchvision import models

OUT_DIR = "/content/model_v4"
IMG_SIZE = 224

class LungClassifierV3(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.efficientnet_b2(weights=None)
        n_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n_feat, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 3)
        )
    def forward(self, x):
        return self.backbone(x)

model = LungClassifierV3()
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location="cpu"))
model.eval()
print("✓ Model loaded")

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model, dummy, f"{OUT_DIR}/model_v4.onnx",
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17
)
with open(f"{OUT_DIR}/classes_v4.json", "w") as f:
    json.dump(["Normal", "Pneumonia", "ILD"], f)

fsize = os.path.getsize(f"{OUT_DIR}/model_v3.onnx") / 1e6
print(f"✓ model.onnx ({fsize:.1f} MB)")

# Copy to Google Drive
# DRIVE_OUT = "/content/drive/MyDrive/lung_model"
# os.makedirs(DRIVE_OUT, exist_ok=True)
# shutil.copy(f"{OUT_DIR}/model_v3.onnx", f"{DRIVE_OUT}/model_v3.onnx")
# shutil.copy(f"{OUT_DIR}/classes_v3.json", f"{DRIVE_OUT}/classes_v3.json")
# shutil.copy(f"{OUT_DIR}/best_model.pt", f"{DRIVE_OUT}/best_model.pt")
# print(f"✓ Saved to Google Drive: {DRIVE_OUT}/")

In [ ]:
import os, json, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import time, random

CLASSES = ["Normal", "Pneumonia", "ILD"]
IMG_SIZE = 224
BATCH = 64
EPOCHS = 50
LR = 2e-4
PATIENCE = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/data/data"
OUT_DIR = "/content/model_v4"
os.makedirs(OUT_DIR, exist_ok=True)

class CXRDataset(Dataset):
    def __init__(self, paths, labels, tfm=None):
        self.paths, self.labels, self.tfm = paths, labels, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.labels[i]

# KEY CHANGE 1: Medical-imaging specific augmentation
# Less color jitter (X-rays are grayscale), more geometric transforms
train_tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.1))
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# KEY CHANGE 2: Use MORE ILD and Normal images for better distinction
# More data = more patterns to distinguish between the two
def load_balanced(data_dir):
    class_paths = {}
    for cls in CLASSES:
        d = os.path.join(data_dir, cls)
        files = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.png')]
        class_paths[cls] = files
        print(f"  {cls}: {len(files)} raw images")

    # Normal: use 5000 (more examples of what "healthy" looks like)
    # ILD: use 5000 (more examples of disease patterns)
    # Pneumonia: oversample 322 → 3000
    TARGETS = {"Normal": 5000, "Pneumonia": 3000, "ILD": 5000}

    paths, labels = [], []
    for i, cls in enumerate(CLASSES):
        files = class_paths[cls]
        target = TARGETS[cls]
        random.seed(42)
        if len(files) >= target:
            files = random.sample(files, target)
        else:
            repeats = target // len(files)
            remainder = target % len(files)
            files = files * repeats + random.sample(files, remainder)
        paths += files
        labels += [i] * len(files)
        print(f"  {cls}: {len(files)} after balancing (target={target})")
    return paths, labels

print(f"Device: {DEVICE}")
print("Loading data...")
paths, labels = load_balanced(DATA_DIR)

tr_p, va_p, tr_l, va_l = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"\nTrain: {len(tr_p)} | Val: {len(va_p)}")

counts = Counter(tr_l)
weights = [1.0 / counts[l] for l in tr_l]
sampler = WeightedRandomSampler(weights, len(weights))

tr_dl = DataLoader(CXRDataset(tr_p, tr_l, train_tfm), BATCH, sampler=sampler, num_workers=2, pin_memory=True)
va_dl = DataLoader(CXRDataset(va_p, va_l, val_tfm), BATCH, shuffle=False, num_workers=2, pin_memory=True)

# KEY CHANGE 3: EfficientNet-B3 (bigger backbone, more features to distinguish subtle differences)
class LungClassifierV4(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        n_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n_feat, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, n_cls)
        )
    def forward(self, x):
        return self.backbone(x)

model = LungClassifierV4().to(DEVICE)

# KEY CHANGE 4: Focal Loss — focuses on HARD examples (Normal vs ILD confusion)
# Standard CrossEntropy treats all mistakes equally
# Focal Loss puts MORE weight on the ones the model gets wrong
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # class weights

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# KEY CHANGE 5: Slightly higher weight for Normal and ILD since that's the confusion
class_weights = torch.tensor([1.2, 1.0, 1.2], dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(gamma=2.0, alpha=class_weights)

# Phase 1: Train head only (freeze backbone)
print("\n── Phase 1: Training head only (5 epochs) ──")
for param in model.backbone.features.parameters():
    param.requires_grad = False

head_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
for epoch in range(5):
    model.train()
    correct, total = 0, 0
    for imgs, lbls in tr_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        head_optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        head_optimizer.step()
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    print(f"  Head epoch {epoch+1}/5  train_acc={correct/total:.4f}")

# Phase 2: Unfreeze and fine-tune
print("\n── Phase 2: Fine-tuning full model ──")
for param in model.backbone.features.parameters():
    param.requires_grad = True

# KEY CHANGE 6: Discriminative learning rates
# Lower LR for backbone (pretrained), higher for head (new)
optimizer = optim.AdamW([
    {"params": model.backbone.features.parameters(), "lr": LR * 0.1},
    {"params": model.backbone.classifier.parameters(), "lr": LR}
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

start = time.time()
for epoch in range(EPOCHS):
    ep_start = time.time()
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, lbls in tr_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)

    model.eval()
    va_preds, va_true = [], []
    with torch.no_grad():
        for imgs, lbls in va_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            va_preds.extend(model(imgs).argmax(1).cpu().tolist())
            va_true.extend(lbls.cpu().tolist())

    tr_acc = correct / total
    va_acc = sum(p == t for p, t in zip(va_preds, va_true)) / len(va_true)
    report = classification_report(va_true, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    # KEY CHANGE 7: Track Normal-ILD confusion specifically
    cm = confusion_matrix(va_true, va_preds)
    normal_as_ild = cm[0][2]  # Normal predicted as ILD
    ild_as_normal = cm[2][0]  # ILD predicted as Normal
    confusion_score = normal_as_ild + ild_as_normal

    scheduler.step()
    history["train_loss"].append(running_loss / total)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(macro_f1)

    ep_time = time.time() - ep_start
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={running_loss/total:.4f}  train_acc={tr_acc:.4f}  "
          f"val_acc={va_acc:.4f}  macro_f1={macro_f1:.4f}  "
          f"N↔ILD_confusion={confusion_score}  time={ep_time:.0f}s")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")
        print(f"  ✓ Saved (macro_f1={macro_f1:.4f}, N↔ILD={confusion_score})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

print(f"\nTotal: {(time.time()-start)/60:.1f} min | Best macro F1: {best_f1:.4f}")

# Final report
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model_v4.pt", map_location=DEVICE))
model.eval()
va_preds, va_true = [], []
with torch.no_grad():
    for imgs, lbls in va_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        va_preds.extend(model(imgs).argmax(1).cpu().tolist())
        va_true.extend(lbls.cpu().tolist())

print("\n" + "="*50)
print("Final Classification Report:")
print("="*50)
print(classification_report(va_true, va_preds, target_names=CLASSES, zero_division=0))
cm = confusion_matrix(va_true, va_preds)
print("Confusion Matrix:")
print(f"{'':>12} {'Normal':>8} {'Pneumonia':>10} {'ILD':>8}")
for i, cls in enumerate(CLASSES):
    print(f"{cls:>12} {cm[i][0]:>8} {cm[i][1]:>10} {cm[i][2]:>8}")